# Benchmark: TDVP wall time against N and h

Wall time of the TDVP runs of the error scan in `notebooks/results/01_tdvp_vs_ed_baseline.ipynb` (RK45, `dt = 1e-3`, `T = 2`, dense Jacobian QGT, `FullSumState`, RBM $\alpha = 1$). The times are read from the same cache (`outputs/tdvp_vs_ed_error_scan/`), so run that notebook first, or set `RECOMPUTE = True` here.

The `tstops` grid caps every step at `dt`, so a run makes at least `T / dt = 2000` RK45 steps whatever $h$ is. The exact evolution (`expm_multiply`) takes well under a second at these sizes and is not shown.

In [ ]:
import time

import numpy as np
from IPython.display import Image, display

from QELM.NQS.dynamics import (
    BASELINE_SEED,
    FieldSweep,
    TDVPSettings,
    accumulated_error_vs_field,
    exact_time_evolution,
    time_grid,
)
from QELM.NQS.measurements import all_to_all_xx_hamiltonian
from QELM.NQS.paths import experiment_paths
from QELM.NQS.utils.makeplots import plot_field_scan
from QELM.NQS.utils.outputs import find_repo_root
from QELM.NQS.variational import rbm, spin_half_hilbert, variational_state

ROOT = find_repo_root()
SIZES = (4, 6, 8)
FIELDS = (0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0)
T_SCAN = 2.0
DT = 1e-3
SEED = BASELINE_SEED
SETTINGS = TDVPSettings(dt=DT, integrator="RK45", diag_shift=1e-6, linear_solver="svd")
RECOMPUTE = False

cache = ROOT / "outputs" / "tdvp_vs_ed_error_scan"
sweeps: dict[int, FieldSweep] = {}
for n in SIZES:
    file = cache / f"sweep_N{n}_T{T_SCAN:g}_seed{SEED}.npz"
    if file.is_file() and not RECOMPUTE:
        sweeps[n] = FieldSweep.load(file)
    else:
        sweeps[n] = accumulated_error_vs_field(
            n_spins=n, fields=FIELDS, J=1.0, T=T_SCAN, seed=SEED, settings=SETTINGS
        )
        sweeps[n].save(file)
paths = experiment_paths(
    experiment_name="tdvp_benchmark", outputs_root=ROOT / "outputs"
)

In [ ]:
n_steps = round(T_SCAN / DT)
figure = plot_field_scan(
    {f"N={n}": s.fields for n, s in sweeps.items()},
    {
        f"N={n}": {"wall_time": s.wall_time, "per_step": 1e3 * s.wall_time / n_steps}
        for n, s in sweeps.items()
    },
    panels={"wall_time": "TDVP wall time (s)", "per_step": "ms per grid step"},
    filename="tdvp_wall_time.png",
    directory=paths.plots,
    title=f"TDVP wall time, T={T_SCAN:g}, dt={DT:g}",
)
display(Image(filename=str(figure)))
for n, s in sweeps.items():
    print(f"N={n}: {s.wall_time.min():6.1f} s to {s.wall_time.max():6.1f} s per run")

In [ ]:
# Exact evolution on the same grid, for scale.
for n in SIZES:
    hilbert = spin_half_hilbert(n)
    H = all_to_all_xx_hamiltonian(hilbert, h=1.0, J=1.0)
    psi0 = np.asarray(variational_state(hilbert, rbm(), seed=SEED).to_array())
    start = time.perf_counter()
    exact_time_evolution(H, psi0, time_grid(T_SCAN, DT))
    print(f"N={n}: ED on {n_steps + 1} times in {time.perf_counter() - start:.3f} s")